In [1]:
import json
import re
import pathlib
import multiprocessing as mp
import xarray as xr
import tqdm

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import numpy.ma as ma
from scipy.interpolate import griddata

from functools import partial
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

In [2]:
src_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations')
boundary_path = pathlib.Path('/gpfs/work2/0/prjs1027/guus/data/BoundaryConditions/bnd.nc')
bathymetry_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs/02-2022/geometry/swan-ns-j22_6-v1a_adjust.bot')
write_dir = pathlib.Path('/gpfs/work2/0/prjs1027/tfrecords/run3')
write_dir.mkdir(exist_ok=True, parents=True)
filtered_paths_50 = [path for path in src_path.glob('ens*') if re.search(r'\d+$', str(path))]
# filtered_paths = [path for path in src_path.glob('ens*') if re.search(r'\\b([0-9]|1[0-9]|2[0-4])\\b', str(path))]

In [3]:
filtered_paths = []
for path in filtered_paths_50:
    ens = re.search(r'\d+$', str(path)).group(0)
    if int(ens) in range(30):
        filtered_paths.append(path)
filtered_paths

[PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens28'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens16'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens19'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens17'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens8'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens23'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens9'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens18'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens10'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens22'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens4'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens27'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-runs/02-2022/computations/ens5'),
 PosixPath('/gpfs/work2/0/prjs1027/swn-run

In [4]:
boundary_ds = xr.open_dataset(boundary_path)
dummy_ds = xr.open_dataset(next(filtered_paths[0].glob('*/output/*.nc')))

In [5]:
output_variables = ["hs", "theta0", "tmm10", "tps", "hswe"]
input_variables = ["xwnd", "ywnd", "wl", "xcur", "ycur"]
boundary_variables = ["hs", "hswe", "tmm10"]
conversion_dict = {"hs": "Hm0", "hswe": "HE10", "tmm10": "Tmm10", "theta0": "meanD"}

In [6]:
def load_ascii_bathy(input_grid_file_path, lon_start, lat_start, lon_step, lat_step):
    """
    Loads bathymetry and grid from ascii file (GEBCO)
 
    Args:
        input_grid_file_path (str): path to grid file
        lon_start: start longitude
        lat_start:
    Returns:
        lon, lat, bottom_data (np.array): 2D arrays with lon and lat values
        and bottom data
 
    Example usage:
        lon_start, lat_start, lon_step, lat_step = -3.000000, 51.500000, 0.500000, 0.500000
        lon_bathy, lat_bathy, bottom_data = input_loader.load_bathy(extra_path + "depth.GRD", lon_start, lat_start, lon_step, lat_step)
 
    """
    # Read the bottom grid data from the ASCII file
    bottom_data = np.loadtxt(input_grid_file_path)
 
    # Transpose the data to match the provided organization (latitude columns, longitude rows)
    bottom_data = bottom_data.T
 
    # Replace -999 with NaN
    bottom_data[bottom_data == -999] = np.nan
 
    lons = np.arange(lon_start, lon_start + len(bottom_data[0]) * lon_step, lon_step)
    lats = np.arange(lat_start, lat_start + len(bottom_data) * lat_step, lat_step)
 
    # Create a meshgrid from the coordinates
    lon, lat = np.meshgrid(lons, lats)
 
    return lon, lat, bottom_data

bathymetry = load_ascii_bathy(bathymetry_path, -12, 48, 0.05, 1/30)[2]

dummy_ds['wl'] = dummy_ds['ssh'] + bathymetry

In [7]:
def correct_nan_theta0(ds):
    for i in range(ds.time.shape[0]):
        hs = ds['hs'].values[i, ...]
        theta0 = ds['theta0'].values[i, ...]
        mask_hs = ma.masked_invalid(hs).mask
        mask_theta0 = ma.masked_invalid(theta0).mask
        
        invalid_ns_mask = np.logical_xor(mask_hs, mask_theta0)
        if invalid_ns_mask.sum() > 0:
            # Extract valid data points
            x = np.arange(theta0.shape[1])
            y = np.arange(theta0.shape[0])
            
            xx, yy = np.meshgrid(x, y)
            x1 = xx[~invalid_ns_mask]
            y1 = yy[~invalid_ns_mask]
            
            new_data = theta0[~invalid_ns_mask]
                    
            # Create interpolator for valid data
            interpolated = griddata((x1, y1), new_data.ravel(), (xx, yy), method='nearest', fill_value=None)
            ds['theta0'].values[i, ...] = interpolated
    return ds

def correct_outlier_cur(ds):
    for i in range(ds.time.shape[0]):
        xcur = ds['xcur'].values[i, ...]
        ycur = ds['ycur'].values[i, ...]
        
        outliers_xcur = np.abs(xcur) > 3
        outliers_ycur = np.abs(ycur) > 3

        xcur[outliers_xcur] = np.nanmean(xcur[~outliers_xcur])
        ycur[outliers_ycur] = np.nanmean(ycur[~outliers_ycur])
        
        ds['xcur'].values[i, ...] = xcur
        ds['ycur'].values[i, ...] = ycur

    return ds
        

In [8]:
# %%
def compute_min_max(path):
    all_datasets = []
    min_values = {}
    max_values = {}
    for var in dummy_ds.variables:
        min_values[var] = []
        max_values[var] = []
    
    for _ in tqdm.tqdm(list(path.glob('*'))):
        try:
            ds_path = next(_.glob('output/*.nc'))
            ds = xr.open_dataset(ds_path)

            ds['wl'] = ds['ssh'] + bathymetry
            ds = correct_outlier_cur(ds)
            
            for var in ds.variables:
                min_values[var].append(ds[var].min().values)
                max_values[var].append(ds[var].max().values)
        except:
            continue

    min_max_values = {}
    for var in dummy_ds.variables:
        min_value = np.min(min_values[var])
        max_value = np.max(max_values[var])
        min_max_values[var] = np.array([min_value, max_value])
    return min_max_values


In [13]:
min_max_values = {}
for paths in filtered_paths:
    min_max_values[paths.stem] = compute_min_max(paths)

100%|██████████| 364/364 [03:35<00:00,  1.69it/s]


In [14]:
min_max_values_detailed = {}
for ens in min_max_values.keys():
    for var in min_max_values[ens].keys():
        try:
            min_max_values_detailed[var] = np.concatenate([min_max_values_detailed[var], min_max_values[ens][var]])
        except:
            min_max_values_detailed[var] = min_max_values[ens][var]


In [15]:
for var in min_max_values_detailed.keys():
    min_value = np.nanmin(min_max_values_detailed[var])
    max_value = np.nanmax(min_max_values_detailed[var])
    min_max_values_detailed[var] = {
        "minimum": min_value,
        "maximum": max_value
    }


In [16]:
df = pd.DataFrame(min_max_values_detailed)

In [17]:
ds_min_max = xr.Dataset.from_dataframe(df)
ds_min_max_dir = write_dir / "min-max-values"
ds_min_max_dir.mkdir(exist_ok=True, parents=True)
ds_min_max.to_netcdf(ds_min_max_dir / "min-max-values.nc")